In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout,LSTM,Input
import datetime
from pathlib import Path



In [2]:
def filter_to_shared_range(df,df_bounds,val_cols):
    parts = []
    df_bounds = df_bounds.reset_index()
    for _,row in df_bounds.iterrows():
        activity = row['activity_code']
        t_min = row['min_timestamp']
        t_max = row['max_timestamp']

        mask = (df['activity_code'] == activity) & (df['timestamp'] >= t_min) & (df['timestamp'] <= t_max)

        parts.append(df.loc[mask,['user_id','activity_code','timestamp']+val_cols])
    return pd.concat(parts,ignore_index=True)

    


In [3]:


def filter_and_merge_sensors(accel_path, gyro_path):
    acc_column_names = ['user_id','activity_code','timestamp','ax','ay','az']
    gy_column_names = ['user_id','activity_code','timestamp','gx','gy','gz']
    acc_df = pd.read_csv(accel_path,header=None, names=acc_column_names)
    gy_df =  pd.read_csv(gyro_path,header=None, names=gy_column_names)

    acc_bounds = acc_df.groupby('activity_code')['timestamp'].agg(['min','max'])
    gy_bounds = gy_df.groupby('activity_code')['timestamp'].agg(['min','max'])

    bounds = acc_bounds.join(gy_bounds,lsuffix='_acc',rsuffix='_gy')

    df_bounds = pd.DataFrame(index=bounds.index)
    df_bounds['min_timestamp'] = bounds[['min_acc','min_gy']].max(axis=1)
    df_bounds['max_timestamp'] = bounds[['max_acc','max_gy']].min(axis=1)


    acc_filtered = filter_to_shared_range(acc_df,df_bounds,['ax','ay','az'])
    gy_filtered = filter_to_shared_range(gy_df,df_bounds,['gx','gy','gz'])


    merged_df = pd.merge(
        acc_filtered,
        gy_filtered.drop(columns='user_id'),
        on=[ 'activity_code', 'timestamp'],
    )

    merged_df = merged_df.sort_values(['activity_code','timestamp']).reset_index(drop=True)

    return merged_df

In [4]:
accel_path = Path('phone/accel')
acc_file_names = os.listdir(accel_path)
df = pd.concat((filter_and_merge_sensors(f'phone/accel/{filename}',f'phone/gyro/{filename.replace("accel","gyro")}')  for filename in acc_file_names ))
df = df.sort_values(['user_id','activity_code'],ascending=[True,True])

In [5]:
df = df.astype(str).apply(lambda s: s.str.replace(';','',regex=False).str.strip())

numeric_cols = [
    'user_id',
    'timestamp',
    'ax', 'ay', 'az',
    'gx', 'gy', 'gz'
]

df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

In [66]:
df.to_csv("merged_data.csv",index=False)


In [6]:
df.head()

,user_id,activity_code,timestamp,ax,ay,az,gx,gy,gz
0,1600,A,252207918580802,-4.332779,13.361191,-0.718872,-0.853210,0.297226,0.890182
1,1600,A,252207968934806,-0.319443,13.318359,-0.232025,-0.875137,0.015472,0.162231
2,1600,A,252208019288809,1.566452,9.515274,-0.017776,-0.720169,0.388489,-0.284012
3,1600,A,252208069642813,-0.323746,5.262665,0.322342,-0.571640,1.227402,-0.241669
4,1600,A,252208119996817,-1.811676,3.710510,1.373932,-0.380493,1.202835,-0.213135


In [67]:
df.shape

(2909149, 9)

In [78]:
new_df = pd.read_csv("merged_data.csv")

In [79]:
def create_windows(dataframe, window_size=100, step_size=50):
    X_list = []
    y_list = []


    for i in range(0, len(dataframe) - window_size, step_size):
        window = dataframe.iloc[i : i + window_size]

        # Only accept the window if the activity remains identical across the window span
        if window['activity_code'].nunique() == 1 and window['user_id'].nunique() == 1:
            features = window[['ax', 'ay', 'az','gx','gy','gz']].values
            label = window['activity_code'].iloc[0]

            X_list.append(features)
            y_list.append(label)

    return np.array(X_list), np.array(y_list)

In [80]:
X_raw, y_raw = create_windows(new_df,180,180)

In [81]:
X_raw.shape

(15373, 180, 6)

found out the correctness of windows

In [ ]:
count = 0
for j in range(len(X_raw)):
    user_id = []
    
    for i in range(len(X_raw[j])):
        user_id.append(X_raw[j][i][0])
        
    new_set = set(user_id)
    if len(new_set) != 1:
        print("anomaly found")

In [82]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
num_classes = len(np.unique(y_encoded))

# Convert numerical integers to One-Hot binary matrices (e.g., 2 -> [0, 0, 1])
y_categorical = to_categorical(y_encoded, num_classes=num_classes)

# Stratified split to keep activity distributions equal in train/test splits
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
)

window_count_train,timesteps_train,feature_count_train = X_train.shape
window_count_test,timesteps_test,feature_count_test = X_test.shape

X_train_2D = X_train.reshape(-1,feature_count_train)
X_test_2D = X_test.reshape(-1,feature_count_test)

scaler = StandardScaler()
scaler.fit(X_train_2D)
X_train_scaled = scaler.transform(X_train_2D)
X_test_scaled = scaler.transform(X_test_2D)

X_train_scaled = X_train_scaled.reshape(window_count_train,timesteps_train,feature_count_train)
X_test_scaled = X_test_scaled.reshape(window_count_test,timesteps_test,feature_count_test)

# Extract shape configurations dynamically
time_steps = X_train_scaled.shape[1]  
num_features = X_train_scaled.shape[2]  



In [101]:
print("Compiling 1D-CNN Model...")
model = Sequential([
    # First Convolutional block to extract local spatial-temporal features
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(time_steps, num_features)),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    # Second Convolutional block for deeper hierarchical patterns
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    # Flatten the 3D feature maps to 1D vectors for dense categorization
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),

    # Output layer using Softmax to get probability distribution over activities
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Compiling 1D-CNN Model...


c:\Users\David\miniconda3\envs\ai_env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_9 (Conv1D)               │ (None, 178, 64)        │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 176, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 88, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 88, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 86, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_7 (MaxPooling1D)  │ (None, 43, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 43, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 5504)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       704,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 18)             │         2,322 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 745,234 (2.84 MB)

 Trainable params: 745,234 (2.84 MB)

 Non-trainable params: 0 (0.00 B)

In [102]:
print("Starting network training...")
history = model.fit(x=X_train_scaled, y=y_train, epochs=30,verbose=1)

# Final validation evaluation
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\n==========================================")
print(f"Training Complete!")
print(f"Model Accuracy on Test Set: {test_acc * 100:.2f}%")
print(f"==========================================")

# Display mapping index reference
print("\nClass Mapping Reference:")
for index, class_label in enumerate(label_encoder.classes_):
    print(f"Index {index} corresponds to Activity Code: {class_label}")

Starting network training...
Epoch 1/30
385/385 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.1786 - loss: 2.3936
Epoch 2/30
385/385 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.2572 - loss: 2.0707
Epoch 3/30
385/385 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.3091 - loss: 1.9307
Epoch 4/30
385/385 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.3239 - loss: 1.8601
Epoch 5/30
350/385 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3529 - loss: 1.7870

KeyboardInterrupt: 